## Łączenie wszytkich plików .nc

In [2]:
import xarray as xr
import glob

sciezka = "./dataset/era5/krakow_temp_*.nc" 
pliki = sorted(glob.glob(sciezka))

ds = xr.open_mfdataset(pliki, combine='by_coords', chunks={'valid_time': 1000})

ds.to_netcdf('./dataset/krakow_era5_2000_2025.nc')

print(ds)

<xarray.Dataset> Size: 27MB
Dimensions:     (valid_time: 227928, latitude: 4, longitude: 6)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2MB 2000-01-01 ... 2025-12-31T23:...
  * latitude    (latitude) float64 32B 50.2 50.1 50.0 49.9
  * longitude   (longitude) float64 48B 19.7 19.8 19.9 20.0 20.1 20.2
    number      int64 8B 0
    expver      (valid_time) <U4 4MB dask.array<chunksize=(744,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 22MB dask.array<chunksize=(744, 4, 6), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-11T14:34 GRIB to CDM+CF via cfgrib-0.9.1...


## Wyodrębnienie danych z pliku NetCDF

In [15]:
print("Wczytywanie ERA5...")
ds = xr.open_dataset('./dataset/krakow_era5_2000_2025.nc')

# Wybieramy najbliższy punkt dla Balic i Obserwatorium
# Balice: 50.077, 19.784 | Obserwatorium: 50.067, 19.959
era_balice = ds['t2m'].sel(latitude=50.077, longitude=19.784, method='nearest').to_dataframe().reset_index()
era_obs = ds['t2m'].sel(latitude=50.067, longitude=19.959, method='nearest').to_dataframe().reset_index()

# Konwersja K -> C i ujednolicenie nazw kolumn czasu
for df in [era_balice, era_obs]:
    df['temp_era5'] = df['t2m'] - 273.15
    # W ERA5 kolumna czasu często nazywa się 'time' lub 'valid_time'
    # Zmieniamy ją na 'timestamp', żeby łatwo łączyć
    if 'time' in df.columns:
        df.rename(columns={'time': 'timestamp'}, inplace=True)
    elif 'valid_time' in df.columns:
        df.rename(columns={'valid_time': 'timestamp'}, inplace=True)

Wczytywanie ERA5...


## Łączenie wszystkich plików dla stacji Kraków-Balice

In [3]:
import glob
import pandas as pd

# Scalanie wszystkich plików Balic
path = "dataset/imgw/balice/KRAKOW_BALICE_*.csv"
all_files = sorted(glob.glob(path))
df_total = pd.concat([pd.read_csv(f) for f in all_files])

# Opcjonalnie: stwórz kolumnę czasową dla wygody
df_total['dt'] = pd.to_datetime(df_total[['year', 'month', 'day', 'hour']])
df_total.to_csv("./dataset/krakow_balice_imgw_2000_2025.csv", index=False)

## Ujednolicenie danych IMGW do UTC

In [ ]:
def prepare_imgw(path):
    print(f"Przetwarzanie {path}...")
    df = pd.read_csv(path)
    
    # Tworzymy czas lokalny z kolumn rok, mc, dz, gg
    df['dt_local'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    
    # Konwersja Lokalny -> UTC (naprawia błąd AmbiguousTimeError)
    df['timestamp'] = df['dt_local'].dt.tz_localize(
        'Europe/Warsaw', ambiguous='NaT', nonexistent='shift_forward'
    ).dt.tz_convert('UTC').dt.tz_localize(None)
    
    return df.dropna(subset=['timestamp'])[['timestamp', 'temp']]

imgw_balice = prepare_imgw('./dataset/krakow_balice_imgw_2000_2025.csv')
imgw_obs = prepare_imgw('./dataset/krakow_obserwatorium_imgw_2000_2025.csv')

Przetwarzanie ./dataset/krakow_balice_imgw_2000_2025.csv...


KeyError: "['temp_imgw'] not in index"

## Łączenie danych ERA5 oraz IMGW

In [ ]:
# Łączymy po kolumnie 'timestamp'
final_balice = pd.merge(era_balice[['timestamp', 'temp_era5']], imgw_balice, on='timestamp', how='inner')
final_obs = pd.merge(era_obs[['timestamp', 'temp_era5']], imgw_obs, on='timestamp', how='inner')

# Zapisujemy do plików
final_balice.to_csv('./dataset/final_balice.csv', index=False)
final_obs.to_csv('./dataset/final_obserwatorium.csv', index=False)

print("\n✅ Sukces! Pliki zapisane:")
print(f"- Balice: {len(final_balice)} wierszy")
print(f"- Obserwatorium: {len(final_obs)} wierszy")


✅ Sukces! Pliki zapisane:
- Balice: 227901 wierszy
- Obserwatorium: 28305 wierszy


# Feature engineering

In [2]:
import pandas as pd
import requests

def get_elevation_srtm(lat, lon):
    # Używamy API bazy Open-Topo, która agreguje dane SRTM 30m
    url = f"https://api.opentopodata.org/v1/srtm30m?locations={lat},{lon}"
    try:
        response = requests.get(url).json()
        return response['results'][0]['elevation']
    except Exception as e:
        print(f"Błąd pobierania wysokości: {e}")
        return None

# Automatyczne uzupełnienie dla Twoich stacji
coords = {
    "Balice": (50.077, 19.784),
    "Obserwatorium": (50.067, 19.959)
}

elevations = {name: get_elevation_srtm(lat, lon) for name, (lat, lon) in coords.items()}
print(elevations)

{'Balice': 234.0, 'Obserwatorium': 207.0}


## Pobieranie wysokości i pokrycia terenu dla stacji

In [5]:
import rasterio
import pandas as pd

path_dem = "./dataset/krakow_dem.tif"
path_lc = "./dataset/krakow_land_cover.tif"

def get_val(raster_path, lat, lon):
    with rasterio.open(raster_path) as src:
        # Pamiętaj: rasterio używa (longitude, latitude)
        vals = list(src.sample([(lon, lat)]))
        return vals[0][0]

# Koordynaty
balice_coords = (50.077, 19.784)
obs_coords = (50.067, 19.959)

# POBIERANIE - sprawdzamy każdą stację w obu plikach
data = {
    'station': ['Balice', 'Obserwatorium'],
    'lat': [balice_coords[0], obs_coords[0]],
    'lon': [balice_coords[1], obs_coords[1]],
    'altitude': [
        get_val(path_dem, *balice_coords),
        get_val(path_dem, *obs_coords)
    ],
    'land_cover': [
        get_val(path_lc, *balice_coords),
        get_val(path_lc, *obs_coords)
    ]
}

df_metadata = pd.DataFrame(data)
print(df_metadata)

         station     lat     lon    altitude  land_cover
0         Balice  50.077  19.784  237.412857          30
1  Obserwatorium  50.067  19.959  210.000015          50


## Przygorowanie końcowego zbioru 

In [11]:
import pandas as pd
import numpy as np

# 1. Wczytujemy zsynchronizowane dane (te z UTC)
balice_ts = pd.read_csv('./dataset/final_balice.csv')
obs_ts = pd.read_csv('./dataset/final_obserwatorium.csv')

# 2. Zmiana nazwy kolumny 'temp' na 'temp_imgw'
balice_ts = balice_ts.rename(columns={'temp': 'temp_imgw'})
obs_ts = obs_ts.rename(columns={'temp': 'temp_imgw'})

# 3. Dodajemy cechy geograficzne (te, które wyciągnęłaś z rastrów)
balice_ts['altitude'] = 237.41
balice_ts['land_cover'] = 30
balice_ts['is_urban'] = 0

obs_ts['altitude'] = 210.00
obs_ts['land_cover'] = 50
obs_ts['is_urban'] = 1

# 4. Łączymy wszystko w jedną tabelę
df_final = pd.concat([balice_ts, obs_ts], ignore_index=True)

# 5. DODAJEMY CECHY CZASOWE (Kodowanie cykliczne)
df_final['timestamp'] = pd.to_datetime(df_final['timestamp'])
df_final['hour'] = df_final['timestamp'].dt.hour
df_final['month'] = df_final['timestamp'].dt.month

# Koło czasu: żeby model wiedział, że 23:00 i 00:00 są blisko siebie
df_final['hour_sin'] = np.sin(2 * np.pi * df_final['hour'] / 24)
df_final['hour_cos'] = np.cos(2 * np.pi * df_final['hour'] / 24)
df_final['month_sin'] = np.sin(2 * np.pi * df_final['month'] / 12)
df_final['month_cos'] = np.cos(2 * np.pi * df_final['month'] / 12)

# 6. Ostateczne czyszczenie - usuwamy zbędne kolumny pomocnicze
# Zostawiamy: timestamp, temp_era5, altitude, land_cover, is_urban, hour_sin, hour_cos, month_sin, month_cos i target: temp_imgw
columns_to_keep = [
    'timestamp', 'temp_era5', 'temp_imgw', 'altitude', 'land_cover', 'is_urban', 
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]
df_final = df_final[columns_to_keep]

# 7. Zapisujemy ostateczny plik
df_final.to_csv('finall_dataset.csv', index=False)

print("Plik finall_dataset.csv został wygenerowany pomyślnie!")
print(f"Liczba próbek: {len(df_final)}")
print(df_final.head())

Plik finall_dataset.csv został wygenerowany pomyślnie!
Liczba próbek: 256206
            timestamp  temp_era5  temp_imgw  altitude  land_cover  is_urban  \
0 2000-01-01 00:00:00  -8.532562      -12.0    237.41          30         0   
1 2000-01-01 01:00:00  -8.527802      -12.5    237.41          30         0   
2 2000-01-01 02:00:00  -8.966553      -12.0    237.41          30         0   
3 2000-01-01 03:00:00  -8.621613      -11.0    237.41          30         0   
4 2000-01-01 04:00:00  -8.467133       -9.6    237.41          30         0   

   hour_sin  hour_cos  month_sin  month_cos  
0  0.000000  1.000000        0.5   0.866025  
1  0.258819  0.965926        0.5   0.866025  
2  0.500000  0.866025        0.5   0.866025  
3  0.707107  0.707107        0.5   0.866025  
4  0.866025  0.500000        0.5   0.866025  


In [12]:
correlation = df_final['temp_era5'].corr(df_final['temp_imgw'])
print(f"Korelacja ERA5 vs IMGW: {correlation:.4f}")

Korelacja ERA5 vs IMGW: 0.9731
